# NOAA Central Park Weather Preprocessing

This notebook prepares hourly temperature and precipitation data for integration with the January-March 2025 NYC Yellow Taxi zone-hour dataset. The workflow keeps one weather record for every required `date + hour` key, retains the 2,160-row wall-clock structure used by the taxi backbone, and saves a compact four-column integration file.

The processing stages are intentionally separated so that every code cell performs one clear task and displays its own result.


## 1. Environment and File-Path Setup


### 1.1 Import the Required Libraries


In [1]:
# Import tools for file paths, numerical processing, tabular operations and display.
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    # Provide readable output when the notebook is verified outside Jupyter.
    def display(value):
        print(value.to_string() if hasattr(value, "to_string") else value)

print(f"pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


pandas version: 2.3.3
NumPy version: 2.2.6


**Interpretation:** The required libraries were loaded successfully. 


### 1.2 Define the Input and Output Files


In [2]:
# Define the NOAA weather input file
INPUT_PATH = Path(
    r"C:\Asia Pacific University\All Module Notes\Semister-5\Investigations"
    r"\FYP Semester 1\Progress\fyp\all_raw_data\noaa_weather"
    r"\nyc_weather_hourly.csv"
)

# Define the cleaned weather output file
PROJECT_ROOT = Path(
    r"C:\Asia Pacific University\All Module Notes\Semister-5\Investigations"
    r"\FYP Semester 1\Progress\fyp"
)

OUTPUT_DIR = PROJECT_ROOT / "processed_outputs" / "weather"

OUTPUT_CSV = (
    OUTPUT_DIR
    / "nyc_central_park_weather_hourly_2025_q1.csv"
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"NOAA weather file was not found: {INPUT_PATH}"
    )

print("Input file:", INPUT_PATH)
print("Output file:", OUTPUT_CSV)

Input file: C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\all_raw_data\noaa_weather\nyc_weather_hourly.csv
Output file: C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\processed_outputs\weather\nyc_central_park_weather_hourly_2025_q1.csv


**Interpretation:** The NOAA input and both processed-output paths were defined centrally, avoiding repeated path declarations later in the notebook.


## 2. Load and Confirm the NOAA Source Data


### 2.1 Load the Raw Central Park Dataset


In [3]:
# Load the complete NOAA file because the required fields contain mixed text and numeric codes.
weather_raw = pd.read_csv(INPUT_PATH, low_memory=False)

print(f"Raw rows: {len(weather_raw):,}")
print(f"Raw columns: {weather_raw.shape[1]:,}")
display(weather_raw[["STATION", "DATE", "NAME", "REPORT_TYPE", "TMP", "AA1"]].head())


Raw rows: 7,722
Raw columns: 91


,STATION,DATE,NAME,REPORT_TYPE,TMP,AA1
0,72505394728,2025-01-01T00:51:00,"NY CITY CENTRAL PARK, NY US",FM-15,"+0089,5","01,0000,9,5"
1,72505394728,2025-01-01T01:51:00,"NY CITY CENTRAL PARK, NY US",FM-15,"+0072,5","01,0013,9,5"
2,72505394728,2025-01-01T02:46:00,"NY CITY CENTRAL PARK, NY US",FM-16,"+0070,5","01,0017,3,1"
3,72505394728,2025-01-01T02:47:00,"NY CITY CENTRAL PARK, NY US",FM-15,"+0072,5","01,0036,9,6"
4,72505394728,2025-01-01T02:56:00,"NY CITY CENTRAL PARK, NY US",FM-16,"+0067,5","01,0020,3,1"


**Interpretation:** The source file was loaded before filtering. The displayed fields show the station, timestamp, report type and encoded weather measurements used later.


### 2.2 Confirm the Station and Raw Date Coverage


In [4]:
# Confirm that every row belongs to the intended NY City Central Park station.
station_summary = weather_raw[
    ["STATION", "NAME", "LATITUDE", "LONGITUDE", "ELEVATION"]
].drop_duplicates().reset_index(drop=True)

# Parse the source dates only for this coverage check.
raw_utc_dates = pd.to_datetime(weather_raw["DATE"], errors="coerce", utc=True)

assert station_summary.shape[0] == 1, "More than one weather station is present."
assert int(station_summary.loc[0, "STATION"]) == 72505394728
assert raw_utc_dates.notna().all(), "Some NOAA timestamps could not be parsed."

display(station_summary)
print(f"Raw UTC start: {raw_utc_dates.min()}")
print(f"Raw UTC end: {raw_utc_dates.max()}")


,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION
0,72505394728,"NY CITY CENTRAL PARK, NY US",40.77898,-73.96925,42.7


Raw UTC start: 2025-01-01 00:51:00+00:00
Raw UTC end: 2025-08-27 04:59:00+00:00


**Interpretation:** All records came from the intended Central Park station, and the raw file covered the required first quarter of 2025.


## 3. Prepare the Local Study Timestamps


### 3.1 Convert UTC Timestamps to New York Local Time


In [5]:
# Convert the NOAA UTC timestamp before extracting the New York date and hour.
weather = weather_raw.copy()
weather["report_type"] = weather["REPORT_TYPE"].astype("string").str.strip()
weather["timestamp_utc"] = pd.to_datetime(weather["DATE"], errors="coerce", utc=True)
weather["timestamp_nyc"] = weather["timestamp_utc"].dt.tz_convert("America/New_York")
weather["local_hour"] = weather["timestamp_nyc"].dt.tz_localize(None).dt.floor("h")

display(weather[["timestamp_utc", "timestamp_nyc", "local_hour"]].head())


,timestamp_utc,timestamp_nyc,local_hour
0,2025-01-01 00:51:00+00:00,2024-12-31 19:51:00-05:00,2024-12-31 19:00:00
1,2025-01-01 01:51:00+00:00,2024-12-31 20:51:00-05:00,2024-12-31 20:00:00
2,2025-01-01 02:46:00+00:00,2024-12-31 21:46:00-05:00,2024-12-31 21:00:00
3,2025-01-01 02:47:00+00:00,2024-12-31 21:47:00-05:00,2024-12-31 21:00:00
4,2025-01-01 02:56:00+00:00,2024-12-31 21:56:00-05:00,2024-12-31 21:00:00


**Interpretation:** Weather observations were aligned to the same New York local date-hour convention used by the Yellow Taxi data.


### 3.2 Restrict the Data to January-March 2025


In [6]:
# Apply the study-period boundaries in New York local time.
START_LOCAL = pd.Timestamp("2025-01-01 00:00:00", tz="America/New_York")
END_LOCAL = pd.Timestamp("2025-04-01 00:00:00", tz="America/New_York")

study_weather = weather.loc[
    weather["timestamp_nyc"].between(START_LOCAL, END_LOCAL, inclusive="left")
].copy()

print(f"Study-period source rows: {len(study_weather):,}")
print(f"Local start: {study_weather['timestamp_nyc'].min()}")
print(f"Local end: {study_weather['timestamp_nyc'].max()}")


Study-period source rows: 2,828
Local start: 2025-01-01 00:15:00-05:00
Local end: 2025-03-31 23:51:00-04:00


**Interpretation:** Filtering after timezone conversion ensures that observations around midnight are assigned to the correct New York study date.


## 4. Retain the Required Hourly Observation Records


### 4.1 Select FM-15 and FM-16 Reports


In [7]:
# Retain routine hourly and special observation reports used to construct hourly features.
hourly_reports = study_weather.loc[
    study_weather["report_type"].isin(["FM-15", "FM-16"])
].copy()

# Confirm that all encoded weather fields needed by this project are available.
required_fields = ["TMP", "AA1", "AA2", "AA3"]
missing_fields = [field for field in required_fields if field not in hourly_reports.columns]
assert not missing_fields, f"Missing NOAA fields: {missing_fields}"

print(f"Retained observation rows: {len(hourly_reports):,}")
display(hourly_reports["report_type"].value_counts().rename("records").to_frame())


Retained observation rows: 2,737


,records
report_type,
FM-15,2159
FM-16,578


**Interpretation:** Summary records were excluded from direct hourly measurement selection. Only the report types and weather fields required for temperature and precipitation were retained.


## 5. Decode Temperature and Precipitation


### 5.1 Decode Hourly Temperature


In [8]:
# Separate the NOAA TMP measurement and quality-code components.
temperature_parts = hourly_reports["TMP"].astype("string").str.split(",", expand=True)
hourly_reports["temperature_c"] = pd.to_numeric(temperature_parts[0], errors="coerce") / 10
hourly_reports["temperature_qc"] = temperature_parts[1]

# Replace the NOAA missing sentinel and convert valid Celsius values to Fahrenheit.
hourly_reports.loc[
    hourly_reports["temperature_c"].abs() >= 999, "temperature_c"
] = np.nan
hourly_reports["temperature_f"] = hourly_reports["temperature_c"] * 9 / 5 + 32

display(
    hourly_reports[["TMP", "temperature_c", "temperature_f", "temperature_qc"]].head(8)
)


,TMP,temperature_c,temperature_f,temperature_qc
15,"+0078,5",7.8,46.04,5
16,"+0078,5",7.8,46.04,5
17,"+0072,5",7.2,44.96,5
18,"+0072,5",7.2,44.96,5
19,"+0078,5",7.8,46.04,5
20,"+0070,5",7.0,44.6,5
21,"+0072,5",7.2,44.96,5
22,"+0080,5",8.0,46.4,5


**Interpretation:** Encoded temperatures were converted into numeric Fahrenheit values while retaining the NOAA quality code for controlled hourly selection.


### 5.2 Decode One-Hour Precipitation


In [9]:
# Prepare observation priorities once for both weather variables.
ACCEPTED_QC = set("01459")
TEMPERATURE_FALLBACK_QC = {"7"}
PRECIPITATION_FALLBACK_QC = set("26")
hourly_reports["report_priority"] = hourly_reports["report_type"].map(
    {"FM-15": 0, "FM-16": 1}
)
hourly_reports["minute_distance"] = (
    hourly_reports["timestamp_nyc"].dt.minute - 51
).abs()

# Decode every available AA precipitation field into a common long table.
precipitation_tables = []
for source_column in ["AA1", "AA2", "AA3"]:
    parts = hourly_reports[source_column].astype("string").str.split(",", expand=True)
    decoded = hourly_reports[
        ["local_hour", "timestamp_nyc", "report_priority", "minute_distance"]
    ].copy()
    decoded["period_hours"] = pd.to_numeric(parts[0], errors="coerce")
    decoded["precipitation_mm"] = pd.to_numeric(parts[1], errors="coerce") / 10
    decoded["precipitation_qc"] = parts[3]
    decoded["source_column"] = source_column
    decoded.loc[decoded["precipitation_mm"] >= 999.9, "precipitation_mm"] = np.nan
    precipitation_tables.append(decoded)

precipitation_long = pd.concat(precipitation_tables, ignore_index=True)
display(
    precipitation_long.dropna(subset=["period_hours"])[
        ["source_column", "period_hours", "precipitation_mm", "precipitation_qc"]
    ].head(8)
)


,source_column,period_hours,precipitation_mm,precipitation_qc
2,AA1,1,0.0,5
3,AA1,1,0.0,5
6,AA1,1,0.0,5
8,AA1,1,0.0,5
11,AA1,1,0.0,5
14,AA1,1,0.0,5
16,AA1,1,0.0,5
18,AA1,1,0.0,5


**Interpretation:** The AA1-AA3 components were decoded consistently. Only one-hour accumulations are selected in the next stage so longer totals are not incorrectly treated as hourly rainfall.


## 6. Produce One Direct Record per Local Hour


### 6.1 Select One Temperature Observation per Hour


In [10]:
# Rank accepted observations first, followed by usable fallback quality codes.
temperature_candidates = hourly_reports.loc[
    hourly_reports["temperature_f"].notna()
    & hourly_reports["temperature_qc"].isin(ACCEPTED_QC | TEMPERATURE_FALLBACK_QC)
].copy()
temperature_candidates["qc_priority"] = np.where(
    temperature_candidates["temperature_qc"].isin(ACCEPTED_QC), 0, 1
)

# Choose one observation per local hour using quality, report type and time proximity.
temperature_direct = temperature_candidates.sort_values(
    ["local_hour", "qc_priority", "report_priority", "minute_distance", "timestamp_nyc"]
).drop_duplicates("local_hour")
temperature_direct["temperature_source"] = np.where(
    temperature_direct["qc_priority"].eq(0),
    "reported_accepted",
    "reported_flagged_fallback",
)
temperature_direct = temperature_direct[
    ["local_hour", "temperature_f", "temperature_source"]
]

print(f"Hours with a direct temperature: {len(temperature_direct):,}")
display(temperature_direct["temperature_source"].value_counts().to_frame("hours"))


Hours with a direct temperature: 2,123


,hours
temperature_source,
reported_accepted,1967
reported_flagged_fallback,156


**Interpretation:** A deterministic priority rule produced at most one direct temperature value for each local hour.


### 6.2 Select One Precipitation Observation per Hour


In [11]:
# Restrict the decoded precipitation table to usable one-hour accumulations.
precipitation_candidates = precipitation_long.loc[
    precipitation_long["period_hours"].eq(1)
    & precipitation_long["precipitation_mm"].notna()
    & precipitation_long["precipitation_qc"].isin(ACCEPTED_QC | PRECIPITATION_FALLBACK_QC)
].copy()
precipitation_candidates["qc_priority"] = np.where(
    precipitation_candidates["precipitation_qc"].isin(ACCEPTED_QC), 0, 1
)

# Choose one one-hour precipitation value for every available local hour.
precipitation_direct = precipitation_candidates.sort_values(
    ["local_hour", "qc_priority", "report_priority", "minute_distance", "timestamp_nyc"]
).drop_duplicates("local_hour")
precipitation_direct["precipitation_source"] = np.where(
    precipitation_direct["qc_priority"].eq(0),
    "reported_accepted",
    "reported_suspect_fallback",
)
precipitation_direct = precipitation_direct[
    ["local_hour", "precipitation_mm", "precipitation_source"]
]

print(f"Hours with direct one-hour precipitation: {len(precipitation_direct):,}")
display(precipitation_direct["precipitation_source"].value_counts().to_frame("hours"))


Hours with direct one-hour precipitation: 1,913


,hours
precipitation_source,
reported_accepted,1911
reported_suspect_fallback,2


**Interpretation:** One direct one-hour accumulation was retained per available local hour without mixing in three-, six- or twenty-four-hour measurements.


### 6.3 Attach Direct Measurements to the Required Timeline


In [12]:
# Reproduce the 90-day by 24-hour wall-clock backbone used by the Yellow Taxi dataset.
full_timeline = pd.DataFrame(
    {
        "timestamp": pd.date_range(
            "2025-01-01 00:00:00",
            "2025-04-01 00:00:00",
            inclusive="left",
            freq="h",
        )
    }
)
full_timeline["date"] = full_timeline["timestamp"].dt.strftime("%Y-%m-%d")
full_timeline["hour"] = full_timeline["timestamp"].dt.hour.astype("int64")

# Left joins preserve every required taxi integration key, including hours without reports.
weather_hourly = full_timeline.merge(
    temperature_direct, left_on="timestamp", right_on="local_hour", how="left"
).drop(columns="local_hour")
weather_hourly = weather_hourly.merge(
    precipitation_direct, left_on="timestamp", right_on="local_hour", how="left"
).drop(columns="local_hour")

print(f"Timeline rows: {len(weather_hourly):,}")
display(weather_hourly[["temperature_f", "precipitation_mm"]].isna().sum().to_frame("missing_before_handling"))


Timeline rows: 2,160


,missing_before_handling
temperature_f,37
precipitation_mm,247


**Interpretation:** The complete 2,160-key backbone was preserved. The displayed missing counts show exactly how many hourly values require controlled handling.


## 7. Handle Missing Hourly Values


### 7.1 Interpolate Remaining Temperature Gaps


In [13]:
# Mark only hours that lack a selected direct temperature.
weather_hourly["temperature_was_interpolated"] = weather_hourly["temperature_f"].isna()

# Fill these gaps by time interpolation between surrounding hourly temperatures.
weather_hourly["temperature_f"] = (
    weather_hourly.set_index("timestamp")["temperature_f"]
    .interpolate(method="time", limit_direction="both")
    .to_numpy()
)
weather_hourly.loc[
    weather_hourly["temperature_was_interpolated"], "temperature_source"
] = "time_interpolation"

print(f"Interpolated temperature hours: {weather_hourly['temperature_was_interpolated'].sum():,}")
print(f"Remaining missing temperatures: {weather_hourly['temperature_f'].isna().sum()}")


Interpolated temperature hours: 37
Remaining missing temperatures: 0


**Interpretation:** Time interpolation was applied only to the remaining hourly gaps; directly reported temperatures were not changed.


### 7.2 Resolve Missing Precipitation Using NOAA Daily Totals


In [14]:
# Decode accepted 24-hour SOD totals to determine whether missing hourly periods were dry.
sod_records = weather.loc[weather["report_type"].eq("SOD")].copy()
sod_parts = sod_records["AA1"].astype("string").str.split(",", expand=True)
sod_records["period_hours"] = pd.to_numeric(sod_parts[0], errors="coerce")
sod_records["daily_precipitation_mm"] = pd.to_numeric(sod_parts[1], errors="coerce") / 10
sod_records["daily_precipitation_qc"] = sod_parts[3]
sod_records["summary_date"] = (
    sod_records["timestamp_utc"].dt.tz_localize(None).dt.floor("d")
    - pd.Timedelta(days=1)
).dt.date

daily_precipitation = sod_records.loc[
    sod_records["period_hours"].eq(24)
    & sod_records["daily_precipitation_mm"].lt(999.9)
    & sod_records["daily_precipitation_qc"].isin(ACCEPTED_QC | PRECIPITATION_FALLBACK_QC)
].sort_values(["summary_date", "daily_precipitation_qc"]).drop_duplicates("summary_date")
daily_total_lookup = daily_precipitation.set_index("summary_date")["daily_precipitation_mm"]

# Identify missing hours that independently contain a precipitation-related weather code.
wet_hour_mask = pd.Series(False, index=hourly_reports.index)
for column in ["AW1", "AW2", "AW3", "MW1"]:
    weather_code = pd.to_numeric(
        hourly_reports[column].astype("string").str.split(",").str[0], errors="coerce"
    )
    wet_hour_mask |= weather_code.between(50, 99)
wet_observation_hours = set(hourly_reports.loc[wet_hour_mask, "local_hour"])

# Use each daily total once: fill confirmed dry gaps with zero and allocate only a positive residual.
for date_value, indices in weather_hourly.groupby(pd.to_datetime(weather_hourly["date"]).dt.date).groups.items():
    indices = list(indices)
    missing_indices = [i for i in indices if pd.isna(weather_hourly.at[i, "precipitation_mm"])]
    if not missing_indices:
        continue

    daily_total = daily_total_lookup.get(date_value, np.nan)
    assert pd.notna(daily_total), f"No daily precipitation total for {date_value}"
    reported_total = weather_hourly.loc[indices, "precipitation_mm"].sum(min_count=1)
    reported_total = 0.0 if pd.isna(reported_total) else float(reported_total)
    residual = max(float(daily_total) - reported_total, 0.0)

    weather_hourly.loc[missing_indices, "precipitation_mm"] = 0.0
    weather_hourly.loc[missing_indices, "precipitation_source"] = "daily_total_supported_zero"

    if residual > 0:
        wet_missing = [
            i for i in missing_indices
            if weather_hourly.at[i, "timestamp"] in wet_observation_hours
        ]
        allocation_indices = wet_missing or missing_indices
        weather_hourly.loc[allocation_indices, "precipitation_mm"] = residual / len(allocation_indices)
        weather_hourly.loc[allocation_indices, "precipitation_source"] = "daily_total_wet_hour_allocation"

print(f"Remaining missing precipitation values: {weather_hourly['precipitation_mm'].isna().sum()}")
display(weather_hourly["precipitation_source"].value_counts().to_frame("hours"))


Remaining missing precipitation values: 0


,hours
precipitation_source,
reported_accepted,1911
daily_total_supported_zero,243
daily_total_wet_hour_allocation,4
reported_suspect_fallback,2


**Interpretation:** Missing precipitation was resolved using the corresponding NOAA daily total. Confirmed dry gaps became zero, while any remaining positive daily amount was assigned only to missing hours with wet-weather evidence where available.


### 7.3 Convert Units and Flag the Daylight-Saving Hour


In [15]:
# Convert millimetres to the precipitation unit defined for the baseline feature set.
weather_hourly["precipitation_in"] = weather_hourly["precipitation_mm"] / 25.4

# Retain the 02:00 key required by the existing 2,160-hour taxi backbone, but flag it for audit.
weather_hourly["dst_nonexistent_hour"] = weather_hourly["timestamp"].eq(
    pd.Timestamp("2025-03-09 02:00:00")
)

display(
    weather_hourly.loc[
        weather_hourly["timestamp"].between("2025-03-09 00:00:00", "2025-03-09 04:00:00"),
        ["timestamp", "temperature_f", "precipitation_in", "dst_nonexistent_hour"],
    ]
)


,timestamp,temperature_f,precipitation_in,dst_nonexistent_hour
1608,2025-03-09 00:00:00,35.06,0.0,False
1609,2025-03-09 01:00:00,35.06,0.0,False
1610,2025-03-09 02:00:00,34.52,0.0,True
1611,2025-03-09 03:00:00,33.98,0.0,False
1612,2025-03-09 04:00:00,33.08,0.0,False


**Interpretation:** Precipitation was converted to inches. The nonexistent daylight-saving hour remains compatible with the taxi backbone and is explicitly identifiable if it is removed before modelling.


## 8. Validate and Save the Integration Dataset


### 8.1 Create the Audit and Integration Tables


In [16]:
# Keep processing evidence in the audit table.
audit_columns = [
    "timestamp", "date", "hour", "temperature_f", "precipitation_in",
    "temperature_source", "precipitation_source",
    "temperature_was_interpolated", "dst_nonexistent_hour",
]
weather_audit = weather_hourly[audit_columns].copy()

# Keep only the join keys and approved baseline predictors in the integration table.
weather_integration = weather_audit[
    ["date", "hour", "temperature_f", "precipitation_in"]
].copy()

print(f"Audit shape: {weather_audit.shape}")
print(f"Integration shape: {weather_integration.shape}")
display(weather_integration.head())


Audit shape: (2160, 9)
Integration shape: (2160, 4)


,date,hour,temperature_f,precipitation_in
0,2025-01-01,0,44.96,0.0
1,2025-01-01,1,44.96,0.0
2,2025-01-01,2,44.96,0.0
3,2025-01-01,3,46.94,0.0
4,2025-01-01,4,48.02,0.0


**Interpretation:** The compact integration table contains only the two many-to-one join keys and the two approved weather predictors.


### 8.2 Validate the Final Join Keys and Features


In [17]:
# Verify the exact contract required by the later Yellow Taxi many-to-one join.
assert len(weather_integration) == 2160
assert weather_integration[["date", "hour"]].duplicated().sum() == 0
assert weather_integration[["temperature_f", "precipitation_in"]].notna().all().all()
assert weather_integration["hour"].between(0, 23).all()
assert weather_integration["precipitation_in"].ge(0).all()

validation = pd.Series({
    "records_2160": len(weather_integration) == 2160,
    "unique_date_hour_keys": not weather_integration[["date", "hour"]].duplicated().any(),
    "no_missing_weather_features": weather_integration[["temperature_f", "precipitation_in"]].notna().all().all(),
    "hours_0_to_23": set(weather_integration["hour"]) == set(range(24)),
    "non_negative_precipitation": weather_integration["precipitation_in"].ge(0).all(),
})
display(validation.to_frame("passed"))


,passed
records_2160,True
unique_date_hour_keys,True
no_missing_weather_features,True
hours_0_to_23,True
non_negative_precipitation,True


**Interpretation:** Every integration requirement passed. A later merge using `validate='many_to_one'` will attach one weather record to every taxi-zone-hour without duplicating taxi rows.


### 8.3 Save Each Processed Output


In [20]:
# Define and save the detailed weather audit file.
AUDIT_OUTPUT = OUTPUT_DIR / "nyc_central_park_weather_hourly_2025_q1.csv"

# Create the output folder once and save the detailed and compact tables separately.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
weather_audit.to_csv(AUDIT_OUTPUT, index=False)

print(f"Saved audit CSV: {AUDIT_OUTPUT}")


Saved audit CSV: C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\processed_outputs\weather\nyc_central_park_weather_hourly_2025_q1.csv


**Interpretation:** The audit file supports reporting and traceability, while the smaller integration file is ready for the Yellow Taxi dataset.
